## PENDEKATAN 1 INSET VADER WITH RUNTIME SCORE NORMALIZATION (ISV-RSN)


Bagian ini adalah pendekatan 1 dengan perlakuan kata fungsi 1

In [2]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os

# Konfigurasi path
DATA_PATH = '../../../datapreprocessingcopy/data_preprocessing_final.csv'
INSET_PATH = '../../../kamus/inset_final.csv'
OUTPUT_DIR = '../../outputs/RSN'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [3]:
# 2.2 Load Data Preprocessing Final
df = pd.read_csv(DATA_PATH) 

print(f"\nData preprocessing berhasil dimuat: {len(df)} tweet")
print(f"Kolom: {df.columns.tolist()}")
df.head()


Data preprocessing berhasil dimuat: 13192 tweet
Kolom: ['no', 'timestamp', 'teks', 'teks_processed']


,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,tertib media online DPR pemerintah jangan spor...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus evaluasi lagi kebijakan bebas visa utama...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang pengaturan logis apa undang un...
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,bebas suara dapat memang jamin UU tetapi bebas...


In [ ]:
# 2.3 Load Leksikon InSet dan Definisi Function Words
df_inset = pd.read_csv(INSET_PATH)
df_inset['kata'] = df_inset['kata'].astype(str).str.strip().str.lower()

# Definisi kategori kata fungsi
NEGASI_DAN_MODAL = {
    'tidak', 'bukan', 'jangan', 'belum', 'sangat', 'harus', 'wajib',
    'akan', 'sudah', 'sedang', 'telah', 'boleh', 'bisa'
}
KATA_HUBUNG_PREPOSISI = {
    'dan', 'atau', 'tetapi', 'karena', 'jika', 'di', 'ke', 'dari',
    'pada', 'untuk', 'dengan', 'oleh', 'hingga', 'sejak'
}
PRONOMINA_DEMONSTRATIVA = {
    'saya', 'aku', 'dia', 'kami', 'kamu', 'anda', 'ini', 'itu', 'yang'
}
PARTIKEL_KATA_TANYA = {
    'pun', 'sih', 'ya', 'lah', 'kah', 'apa', 'siapa', 'bagaimana'
}

ALL_FUNCTION_WORDS = (
    NEGASI_DAN_MODAL
    | KATA_HUBUNG_PREPOSISI
    | PRONOMINA_DEMONSTRATIVA
    | PARTIKEL_KATA_TANYA
)

NameError: name 'SLA_LEXICON_PATH' is not defined

In [4]:
# 2.4 Diagnostik dan Konfigurasi Ignore Set
# Diagnostik kata fungsi dalam leksikon
df_found = df_inset[df_inset['kata'].isin(ALL_FUNCTION_WORDS)].copy().sort_values('kata')


# Hitung total kata fungsi yang ditemukan di leksikon
total_fw_found = len(df_found)
print(f"[DIAGNOSTIK] Total kata fungsi ditemukan di InSet: {total_fw_found}")


# Tampilkan detailnya
print("\n[DIAGNOSTIK] Rincian kata fungsi di InSet:")
if not df_found.empty:
    print(df_found[['kata', 'skor']].to_string(index=False))
else:
    print("Tidak ditemukan entri kata fungsi di leksikon.")


# Konfigurasi ignore set (hanya yg akan dinetralkan)
IGNORE_CATEGORIES = KATA_HUBUNG_PREPOSISI | PRONOMINA_DEMONSTRATIVA | PARTIKEL_KATA_TANYA
ignore_set_final = set(df_found[df_found['kata'].isin(IGNORE_CATEGORIES)]['kata'])


# Dictionary lexicon
inset_dict = dict(zip(df_inset['kata'], df_inset['skor']))


# Output konfigurasi akhir
print(f"\n[KONFIGURASI] Dictionary InSet siap: {len(inset_dict)} entri.")
print(f"Kata fungsi yang akan dinetralkan (skor=0): {len(ignore_set_final)} kata.")
print(f"Kata fungsi yang dipertahankan (negasi/modal): {total_fw_found - len(ignore_set_final)} kata.")


[DIAGNOSTIK] Total kata fungsi ditemukan di InSet: 22

[DIAGNOSTIK] Rincian kata fungsi di InSet:
  kata  skor
   aku     2
  anda    -4
  anda    -1
   apa    -3
 boleh     2
 bukan    -3
  dari    -3
   dia    -3
 harus    -5
   itu    -2
jangan    -3
karena    -3
  pada    -3
   pun    -3
sangat     3
  saya    -3
 siapa     1
 sudah     3
 tidak    -5
 wajib    -2
    ya     4
  yang    -5

[KONFIGURASI] Dictionary InSet siap: 9071 entri.
Kata fungsi yang akan dinetralkan (skor=0): 13 kata.
Kata fungsi yang dipertahankan (negasi/modal): 9 kata.


In [5]:
# 2.5 Menampilkan daftar kata yang akan dinetralkan
print("\n[DAFTAR] Kata fungsi yang akan dinetralkan (skor=0):")
for word in sorted(ignore_set_final):
    print(f"  - {word}")


[DAFTAR] Kata fungsi yang akan dinetralkan (skor=0):
  - aku
  - anda
  - apa
  - dari
  - dia
  - itu
  - karena
  - pada
  - pun
  - saya
  - siapa
  - ya
  - yang


In [6]:
# 2.6 Fungsi Tokenisasi
def tokenize(text):
    if not isinstance(text, str):
        return []
    return text.split()

df['tokens'] = df['teks_processed'].apply(tokenize)

print(f"[INFO] Tokenisasi selesai. Total token: {df['tokens'].str.len().sum():,}")

[INFO] Tokenisasi selesai. Total token: 235,560


In [7]:
# 2.7 Fungsi Lexicon Matching dengan Ignore Function
def match_lexicon_with_ignore(tokens, lexicon, ignore_set):
    matched_words = []
    ignored_words = []
    unmatched_words = []
    
    for token in tokens:
        token_lower = token.lower()
        if token_lower in ignore_set:
            ignored_words.append(token)
        elif token_lower in lexicon:
            matched_words.append(token)
        else:
            unmatched_words.append(token)
            
    return matched_words, ignored_words, unmatched_words

In [8]:
# 2.8 Penerapan Lexicon Matching
print("\n[PROSES] Menjalankan lexicon matching dengan ignore function...")

df[['matched_words', 'ignored_words', 'unmatched_words']] = df['tokens'].apply(
    lambda x: pd.Series(match_lexicon_with_ignore(x, inset_dict, ignore_set_final))
)

print("[INFO] Lexicon matching selesai.")


[PROSES] Menjalankan lexicon matching dengan ignore function...
[INFO] Lexicon matching selesai.


In [9]:
# 2.9 Perhitungan Statistik
total_words = df['tokens'].str.len().sum()
total_matched = df['matched_words'].str.len().sum()
total_ignored = df['ignored_words'].str.len().sum()
total_unmatched = df['unmatched_words'].str.len().sum()

print("\n[STATISTIK] Hasil Lexicon Matching (Dengan Ignore Function):")
print(f"Total kata             : {total_words:,}")
print(f"Matched di InSet       : {total_matched:,} ({(total_matched/total_words)*100:.2f}%)")
print(f"Ignored (dinetralkan)  : {total_ignored:,} ({(total_ignored/total_words)*100:.2f}%)")
print(f"Unmatched              : {total_unmatched:,} ({(total_unmatched/total_words)*100:.2f}%)")
print(f"Coverage Rate          : {((total_matched + total_ignored)/total_words)*100:.2f}%")


[STATISTIK] Hasil Lexicon Matching (Dengan Ignore Function):
Total kata             : 235,560
Matched di InSet       : 83,100 (35.28%)
Ignored (dinetralkan)  : 8,599 (3.65%)
Unmatched              : 143,861 (61.07%)
Coverage Rate          : 38.93%


In [ ]:
# 2.9.1
# 1. Kumpulkan semua kata unmatched dari dataframe hasil matching
from collections import Counter

all_unmatched = []
for lst in df['unmatched_words']:  
    all_unmatched.extend([w.lower() for w in lst])

# 2. Hitung frekuensi dan ambil top 50
unmatched_freq = Counter(all_unmatched)
top_50_unmatched = unmatched_freq.most_common(50)

# 3. Tampilkan
print("TOP 50 KATA UNMATCHED PALING SERING MUNCUL:")
print(f"{'Kata':<20} | {'Frekuensi':<10}")
print("-" * 35)
for word, freq in top_50_unmatched:
    print(f"{word:<20} | {freq:<10}")

TOP 50 KATA UNMATCHED PALING SERING MUNCUL:
Kata                 | Frekuensi 
-----------------------------------
dpr                  | 9948      
ruu                  | 6107      
ri                   | 5492      
rakyat               | 3217      
di                   | 3158      
dan                  | 3101      
wakil                | 2451      
ini                  | 1902      
reses                | 1746      
?                    | 1613      
untuk                | 1515      
dengan               | 1434      
ke                   | 1365      
pemerintah           | 1249      
tahun                | 1127      
ketua                | 1118      
partai               | 1086      
masa                 | 944       
!                    | 932       
aspirasi             | 918       
jika                 | 765       
uu                   | 754       
akan                 | 704       
ii                   | 703       
pemilu               | 694       
oleh                 | 638       
in

In [11]:
# 2.10 Preview Hasil Matching
print("\n[PREVIEW] 5 Tweet Pertama:")

for i in range(5):
    print(f"\nTweet {i+1}: {df['teks_processed'].iloc[i][:80]}...")
    print(f"  Matched : {df['matched_words'].iloc[i]}")
    print(f"  Ignored : {df['ignored_words'].iloc[i]}")


[PREVIEW] 5 Tweet Pertama:

Tweet 1: ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !...
  Matched : ['ADIL', 'punya', 'kebijakan', 'ingat']
  Ignored : ['yang', 'yang']

Tweet 2: tertib media online DPR pemerintah jangan sporadis apalagi selektif hanya kepada...
  Matched : ['tertib', 'jangan', 'sporadis', 'selektif', 'hanya']
  Ignored : ['yang']

Tweet 3: harus evaluasi lagi kebijakan bebas visa utama untuk negara tiongkok pak ! ! bah...
  Matched : ['harus', 'lagi', 'kebijakan', 'bebas', 'bahaya', 'martabat']
  Ignored : []

Tweet 4: jangan ngambang pengaturan logis apa undang undang...
  Matched : ['jangan', 'undang', 'undang']
  Ignored : ['apa']

Tweet 5: bebas suara dapat memang jamin UU tetapi bebas sebut tidak harus bablas sehingga...
  Matched : ['bebas', 'suara', 'dapat', 'memang', 'jamin', 'bebas', 'tidak', 'harus', 'bablas', 'tabrak']
  Ignored : []


In [12]:
# 2.11 Simpan Output
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = os.path.join(OUTPUT_DIR, 'lexicon_matching_with_ignore.csv')
df.to_csv(output_path, index=False)

print(f"\n[OUTPUT] Data berhasil disimpan ke: {output_path}")


[OUTPUT] Data berhasil disimpan ke: ../../outputs/RSN\lexicon_matching_with_ignore.csv
